# Proyecto 6
## Introducción
Este proyecto tiene fines educativos simulando un proyecto en un empleo real y formal. En este caso actuamos como el analista de la empresa **ConnectaTel**, empresa de telecomunicaciones con operación en *México* y *Colombia*

Debo entregar un reporte donde pueda explicar **cómo los clientes usan realmente los servicios móviles, es decir llamadas y mensajes**

## Objetivo de la empresa
- Detectar *patrones de uso*
- Detectar *comportamientos atipicos*
- Comprender *qué segmentos de clientes muestras necesidades diferenciadas*

## Fuentes de datos
- ***plans.csv***: Planes actuales (precio, minutos incluidos, gb incluidos, costo por extra)
- ***users_latam.csv***: Info del cliente (edad, ciudad, fecha de nacimiento, plan contratado)
- ***usage.csv***: Detalle de uso real: llamdas (duración) y mensajes (longitud)

## Preguntas del negocio
- ¿Qué segmentos de clientes muestran mayor o menor uso de llamadas y mensajes?
- ¿Qué usuarios presentan valores atípicos que puedan indicar comportamientos inusuales, fraude o errores de registro?
- ¿Cómo varía el uso según la edad y el tipo de plan contratado?
- ¿Qué patrones pueden ayudar a diseñar mejores planes, optimizar la oferta y mejorar la satisfacción del cliente?

## Stack del proyecto
- Numpy
- Pandas
- Seaborn
- Matplotlib

# Inicio del análisis

## 1. Importación del stack y upload de archivos a analizar

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
plans = pd.read_csv("../data/plans.csv")
users = pd.read_csv("../data/users_latam.csv")
usage = pd.read_csv("../data/usage.csv")

## 2. Exploración de datasets

In [3]:
# Exploración dataset plans
display(plans)
display(plans.info())

,plan_name,messages_included,gb_per_month,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute
0,Basico,100,5,100,12,1.2,0.08,0.10
1,Premium,500,20,600,25,1.0,0.05,0.07


<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   plan_name          2 non-null      str    
 1   messages_included  2 non-null      int64  
 2   gb_per_month       2 non-null      int64  
 3   minutes_included   2 non-null      int64  
 4   usd_monthly_pay    2 non-null      int64  
 5   usd_per_gb         2 non-null      float64
 6   usd_per_message    2 non-null      float64
 7   usd_per_minute     2 non-null      float64
dtypes: float64(3), int64(4), str(1)
memory usage: 260.0 bytes


None

En Dataset *plans* tenemos 2 tipos de planes: **Básico** y **Premium**.

Dataset define cada plan con lo que incluye y sus caracteristicas: Minutos incluidos, mensajes incluidos pago mensual, costo por gigabyte, gigabytes por mes, costo por mensaje y costo por minuto

In [4]:
# Exploración de dataset users
display(users.head())
display(users.info())

,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
0,10000,Carlos,Garcia,38,Medellín,2022-01-01 00:00:00.000000000,Basico,NaN
1,10001,Mateo,Torres,53,?,2022-01-01 06:34:17.914478619,Basico,NaN
2,10002,Sofia,Ramirez,57,CDMX,2022-01-01 13:08:35.828957239,Basico,NaN
3,10003,Mateo,Ramirez,69,Bogotá,2022-01-01 19:42:53.743435858,Premium,NaN
4,10004,Mateo,Torres,63,GDL,2022-01-02 02:17:11.657914478,Basico,NaN


<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   user_id     4000 non-null   int64
 1   first_name  4000 non-null   str  
 2   last_name   4000 non-null   str  
 3   age         4000 non-null   int64
 4   city        3531 non-null   str  
 5   reg_date    4000 non-null   str  
 6   plan        4000 non-null   str  
 7   churn_date  466 non-null    str  
dtypes: int64(2), str(6)
memory usage: 250.1 KB


None

En dataset **users** tenemos 4000 registros.

Cuenta con errores en el tipo de dato de las columnas *fecha de registro (reg_date)* y *fecha de cancelación (churn_date)*.

Igual contamos con registros faltantes en la columna *city*, por lo que tendremos un posible sesgo al momento de analizar el resumen estadístico sobre las ciudades en las que opera ConnectaTel

In [5]:
# Exploración de dataset usage
display(usage.head())
display(usage.info())

,id,user_id,type,date,duration,length
0,1,10332,call,2024-01-01 00:00:00.000000000,0.09,NaN
1,2,11458,text,2024-01-01 00:06:30.969774244,NaN,39.0
2,3,11777,text,2024-01-01 00:13:01.939548488,NaN,36.0
3,4,10682,call,2024-01-01 00:19:32.909322733,1.53,NaN
4,5,12742,call,2024-01-01 00:26:03.879096977,4.84,NaN


<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        40000 non-null  int64  
 1   user_id   40000 non-null  int64  
 2   type      40000 non-null  str    
 3   date      39950 non-null  str    
 4   duration  17924 non-null  float64
 5   length    22104 non-null  float64
dtypes: float64(2), int64(2), str(2)
memory usage: 1.8 MB


None

El dataset *Usage* tiene 40000 registros.

Tenemos igual el error en el tipo de dato de la columna *date*

Tenemos la columna ***type***, donde indica *call* o *text* de a cuerdo al tipo de servicio utilizado en ese momento de uso.

*duration* hace referencia a la duración de llamada en minutos y *length* a la cantidad de caracteres.

## Identificación de problemas de calidad en los datos

### Valores nulos

In [6]:
print('Registros en dataset usage', usage.shape[0])
print()
print('Valores nulos en dataset usage')
print(usage.isna().sum())

Registros en dataset usage 40000

Valores nulos en dataset usage
id              0
user_id         0
type            0
date           50
duration    22076
length      17896
dtype: int64


Las columnas **duration** y **length** registras tipos de servicios distintos:
- Duration: Mide la **duración** de la llamda en **minutos**
- Length: Mide la **cantidad de caracteres** de los mensajes

Esto nos indica que la ausencia de tanto valores en los registros tiene sentido, por lo que no se deben eliminar dichos registros, sino tratarlos como información confiable.

La columna **date** tiene **.12% de registros faltantes** (50 de 40,000), por lo que **eliminarlos** no representa un riesgo a la calidad de los datos.

In [7]:
print('Registros en dataset users:', users.shape[0])
print()
print('Valores nulos en dataset users')
print(users.isna().sum())

Registros en dataset users: 4000

Valores nulos en dataset users
user_id          0
first_name       0
last_name        0
age              0
city           469
reg_date         0
plan             0
churn_date    3534
dtype: int64


Tenemos una cantidad considerable de valores nulos en la columna **city** (11.7%), considero prudente utilizar esta cantidad de valores nulos con la nueva categoría "unknown", para no perder de vista el error de registro de ciudad donde se ubican esos usuarios.

**churn_date** es la columna que registra la fecha en la que los usuarios abandonan, entonces si los usuarios no han cancelado sus servicios, es lógico que no aparezca algún valor en esta columna.

### Resumenes estadísticos a columnas

Buscamos identificar Sentinels, valores atípicos, fechas fuera del rango (estamos trabajando con datos del año 2024)

**Sentinels**: Valor especial que se usa como marcador para indicar algo específico, normalmente:

- Indicar un valor faltante
- Señalar el final de una secuencia
- Representar un estado especial
- Diferenciar un caso "normal" de uno "especial"

In [8]:
display(users[['user_id', 'age']].describe())
print()
print('Valores en age menores a 0 =', (users['age']<0).sum())
valores_unicos_negativos = users[users['age']<0]['age'].unique()
print('Valores únicos negativos = ', valores_unicos_negativos)

,user_id,age
count,4000.000000,4000.000000
mean,11999.500000,33.739750
std,1154.844867,123.232257
min,10000.000000,-999.000000
25%,10999.750000,32.000000
50%,11999.500000,47.000000
75%,12999.250000,63.000000
max,13999.000000,79.000000



Valores en age menores a 0 = 55
Valores únicos negativos =  [-999]


En la columna age tenemos el sentinel *-999*, indicando valores desconocidos (55 valores desconocidos), este 1.37% de valores desconocidos pueden ser **imputados** con la mediana para no perder los registros completos.

In [9]:
print('Resumen de ciudades y planes')
display(users[['city', 'plan']].describe())
print()
display(users['city'].value_counts())
print('Valores nulos en city:', users['city'].isna().sum())

Resumen de ciudades y planes


,city,plan
count,3531,4000
unique,7,2
top,Bogotá,Basico
freq,808,2595


city
Bogotá      808
CDMX        730
Medellín    616
GDL         450
Cali        424
MTY         407
?            96
Name: count, dtype: int64

Valores nulos en city: 469


Columna **city** tiene una cantidad considerable de valores nulos (469), sin embargo, tiene entre las opciones del catalogo de ciudades un indicador de desconocidos ("?") con 96 registros en esta opción. Con el propósito de no perder los registros considero ideal **integrar** en los registros con valor nulo en esta categoría de ciudades deconocidas

In [10]:
display(usage[['id', 'user_id', 'duration', 'length']].describe())

,id,user_id,duration,length
count,40000.00000,40000.000000,17924.000000,22104.000000
mean,20000.50000,12002.405975,5.202237,52.127398
std,11547.14972,1157.279564,6.842701,56.611183
min,1.00000,10000.000000,0.000000,0.000000
25%,10000.75000,10996.000000,1.437500,37.000000
50%,20000.50000,12013.000000,3.500000,50.000000
75%,30000.25000,13005.000000,6.990000,64.000000
max,40000.00000,13999.000000,120.000000,1490.000000


In [11]:
print('Valores vacios en columnas:')
display(usage[['duration', 'length']].isna().sum())

values_complete_null = usage.shape[0] - (usage['duration'].isna().sum()) - usage['length'].isna().sum()
print('Valores completamente nulos:', values_complete_null)

Valores vacios en columnas:


duration    22076
length      17896
dtype: int64

Valores completamente nulos: 28


Columnas **duration** y **length** tienen 0 como valor mínimo y tenemos más registros de **length** (22104) que **duration** (17924), indicando de manera general que los usuarios utilizan más los mensajes que llamadas (55.26%).

Tenemos 28 valores que no llevan ningún tipo de registro del servicio usado (0.07%), es una cantidad mínima, por lo que podemos **eliminar** los registros sin comprometer la calidad del análisis

In [12]:
display(usage['type'].describe())

count     40000
unique        2
top        text
freq      22092
Name: type, dtype: object

Confirmando, los servicios de **Texto** son los que lideran el uso de servicios de manera general